<a href="https://colab.research.google.com/github/faizanarif2/worker-safety-helmet-detection/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Worker Safety Helmet Detection using YOLOv8

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os

project_path = "/content/drive/MyDrive/helmet_detection_project"

folders = [
    "dataset",
    "training_results",
    "models",
    "predictions"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("Project folders created.")

Project folders created.


In [4]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 8.8 MB/s eta 0:00:00


In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu128
GPU available: True


In [6]:
!nvidia-smi

Fri Sep 25 11:16:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
from ultralytics import YOLO

print("Ultralytics YOLO imported successfully.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics YOLO imported successfully.


In [8]:
import os

dataset_folder = os.path.join(project_path, "dataset")

print("Dataset folder:", dataset_folder)
print("Files:", os.listdir(dataset_folder))

Dataset folder: /content/drive/MyDrive/helmet_detection_project/dataset
Files: ['helmet_dataset_version1.zip']


In [9]:
import zipfile
from pathlib import Path

zip_files = list(Path(dataset_folder).glob("*.zip"))

print("ZIP files:", [file.name for file in zip_files])

assert len(zip_files) == 1, "Upload exactly one dataset ZIP to the dataset folder."

zip_path = zip_files[0]
extract_path = Path("/content/helmet_dataset")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted.")
print("Extracted files:", os.listdir(extract_path))

ZIP files: ['helmet_dataset_version1.zip']
Dataset extracted.
Extracted files: ['README.roboflow.txt', 'test', 'valid', 'README.dataset.txt', 'train', 'data.yaml']


In [10]:
import yaml

yaml_files = list(extract_path.rglob("data.yaml"))

assert len(yaml_files) == 1, "Could not identify exactly one data.yaml file."

data_yaml = yaml_files[0]
dataset_root = data_yaml.parent

with open(data_yaml, "r") as file:
    data = yaml.safe_load(file)

data["path"] = str(dataset_root)
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

with open(data_yaml, "w") as file:
    yaml.safe_dump(data, file, sort_keys=False)

print("Dataset YAML:", data_yaml)
print("Class mapping:", data["names"])

names = data["names"]
class_names = set(names.values()) if isinstance(names, dict) else set(names)

assert class_names == {"person", "helmet", "no_helmet"}

print("All three classes verified.")

Dataset YAML: /content/helmet_dataset/data.yaml
Class mapping: ['helmet', 'no_helmet', 'person']
All three classes verified.


In [11]:
image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

for split in ["train", "valid", "test"]:
    image_folder = dataset_root / split / "images"
    label_folder = dataset_root / split / "labels"

    images = [
        file for file in image_folder.iterdir()
        if file.suffix.lower() in image_extensions
    ]

    labels = list(label_folder.glob("*.txt"))

    print(f"{split}: {len(images)} images, {len(labels)} label files")

train: 221 images, 221 label files
valid: 63 images, 63 label files
test: 32 images, 32 label files


In [12]:

from pathlib import Path
from ultralytics import YOLO
import torch

assert torch.cuda.is_available(), "Enable a GPU in Colab."
assert Path(data_yaml).exists(), "Run the dataset setup cells first."

runs_dir = Path(project_path) / "training_results"
runs_dir.mkdir(parents=True, exist_ok=True)

experiments = {
    "yolov8n_v1": runs_dir / "yolov8n_v1",
    "yolov8n_mosaic05": runs_dir / "yolov8n_mosaic05",
    "yolov8s_v1": runs_dir / "yolov8s_v1"
}

print("GPU:", torch.cuda.get_device_name(0))
print("Dataset:", data_yaml)
print("Training results:", runs_dir)

GPU: Tesla T4
Dataset: /content/helmet_dataset/data.yaml
Training results: /content/drive/MyDrive/helmet_detection_project/training_results


In [13]:

exp1_path = experiments["yolov8n_v1"]

assert not exp1_path.exists(), "This experiment already exists. Check its saved checkpoints."

model1 = YOLO("yolov8n.pt")

model1.train(
    data=str(data_yaml),
    epochs=60,
    imgsz=640,
    batch=8,
    device=0,
    optimizer="SGD",
    lr0=0.01,
    mosaic=1.0,
    close_mosaic=10,
    patience=60,
    seed=42,
    project=str(runs_dir),
    name="yolov8n_v1",
    plots=True,
    save=True
)

print("Experiment 1 completed.")
print("Best model:", exp1_path / "weights/best.pt")

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/helmet_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_v1, nbs=64, nms=None, opset=None, o

In [14]:
exp2_path = experiments["yolov8n_mosaic05"]

assert not exp2_path.exists(), "This experiment already exists. Check its saved checkpoints."

model2 = YOLO("yolov8n.pt")

model2.train(
    data=str(data_yaml),
    epochs=60,
    imgsz=640,
    batch=8,
    device=0,
    optimizer="SGD",
    lr0=0.01,
    mosaic=0.5,
    close_mosaic=10,
    patience=60,
    seed=42,
    project=str(runs_dir),
    name="yolov8n_mosaic05",
    plots=True,
    save=True
)

print("Experiment 2 completed.")
print("Best model:", exp2_path / "weights/best.pt")

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/helmet_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolov8n_mosaic05, nbs=64, nms=None, opset=N